DAAN-545: Google 2018 entrants, followed across employers for five years
Run one cell at a time. Fill in your verified WRDS metadata before extraction.
Candidate cohort: U.S. Google positions starting during 2018 at entry/junior level.
The full pipeline later checks earlier history to exclude pre-2018 Google workers.
Education: retain the project's U.S. bachelor's requirement during preprocessing.
Track ALL accessible employers and countries; the salary model keeps U.S. jobs
for comparability. Advancement uses known seniority across employers/countries.
These are first OBSERVED Google entrants; incomplete profiles cannot prove first hire.
Entry-level codes may include internships. This version includes them unless you
apply a documented internship rule during cohort review; do not call it full-time-only.
Installation: %pip install wrds pandas numpy
Full pipeline also needs scikit-learn statsmodels mlxtend matplotlib.
No actual WRDS records or empirical findings are included.


In [ ]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
# A new output folder prevents mixing the earlier snapshot design with this one.
OUT = Path('revelio_google_2018_entrants')
OUT.mkdir(exist_ok=True)
START_FROM = pd.Timestamp('2018-01-01')
START_BEFORE = pd.Timestamp('2019-01-01')
FOLLOWUP_YEARS = 5
# This covers the latest possible fifth anniversary of a 2018 entrant.
EXTRACT_THROUGH = (START_BEFORE - pd.Timedelta(days=1)) + pd.DateOffset(years=FOLLOWUP_YEARS)
RCID = None               # Verified Google LLC company ID, NOT parent ID.
JUNIOR = [1, 2]            # Verify entry/junior codes; may include internships.
US_JOB = 'United States'   # Verify the source's country value.
# Set True after confirming access to earlier Google history AND all-employer
# histories through EXTRACT_THROUGH. This is coverage confirmation, not a new
# access permission. Setting it True cannot grant access or fill profile gaps.
HISTORY_COVERAGE_CONFIRMED = False


1. Discover the available tables and columns (metadata only).
These outputs help configure the extraction; they are not EDA/statistics.


In [ ]:
import wrds
# Enter credentials through the prompt; never store passwords in this notebook.
db = wrds.Connection()
print([s for s in db.list_libraries() if 'revelio' in s.lower()])
LIB = 'revelio_individual'   # Replace with your authorized library if different.
print(db.list_tables(library=LIB))


In [ ]:
POS_TABLE = ''              # Enter actual Positions table name.
EDU_TABLE = ''              # Enter actual Education table name.
assert POS_TABLE and EDU_TABLE, 'Enter the table names returned above.'
print(db.describe_table(library=LIB, table=POS_TABLE))
print(db.describe_table(library=LIB, table=EDU_TABLE))
# Use company mapping metadata to verify Google LLC and its RCID.
# Native SQL date fields, inclusive end dates and null=ongoing are assumed.
# If only years/months are supplied, adapt the date rules before proceeding.


2. Map source columns. Blank values are deliberate configuration checkpoints.
The names on the left are our output names, not promised source field names.
Keep one documented occupation taxonomy level throughout the study.


In [ ]:
P = {k: '' for k in ['user_id','position_id','rcid','company_name','startdate','enddate',
                     'country','state','title','occupation','seniority','salary']}
E = {k: '' for k in ['user_id','school','degree','major','grad_date']}
# Fill these dictionaries BEFORE the assertion, e.g. P['user_id']='user_id'
# only if that really is the source column. company_name identifies destinations.
assert RCID is not None and all(P.values()) and all(E.values()), 'Complete mappings and RCID.'

def ident(s):
    # Quote validated SQL identifiers; value filters are parameterized separately.
    if not re.fullmatch(r'[A-Za-z_][A-Za-z_0-9]*', s):
        raise ValueError(f'Invalid identifier: {s!r}')
    return '"' + s + '"'

def qualified(table):
    return ident(LIB) + '.' + ident(table)

def select_columns(mapping):
    # Preserve IDs without floating-point precision loss. This is extraction
    # formatting, not cleaning, deduplication or analytical preprocessing.
    ids = {'user_id','position_id','rcid','school'}
    return ', '.join((f'CAST({ident(v)} AS TEXT)' if k in ids else ident(v))
                     + ' AS ' + ident(k) for k,v in mapping.items())

for table, mapping in [(POS_TABLE,P),(EDU_TABLE,E)]:
    actual = set(db.describe_table(library=LIB, table=table)['name'])
    assert set(mapping.values()) <= actual, 'Mapped column not in source table.'
assert HISTORY_COVERAGE_CONFIRMED, 'Confirm earlier history and cross-employer coverage first.'


In [ ]:
# Extract CANDIDATES: positions that START during 2018, not year-end incumbents.
# Do not require the position to survive through December or any later date.
# These rows may include internal moves; first observed entry is checked later.
base_sql = f"""
SELECT {select_columns(P)} FROM {qualified(POS_TABLE)}
WHERE {ident(P['rcid'])} = %(rcid)s
  AND {ident(P['country'])} = %(country)s
  AND {ident(P['seniority'])} IN %(junior)s
  AND {ident(P['startdate'])} >= %(start_from)s
  AND {ident(P['startdate'])} < %(start_before)s
"""
base = db.raw_sql(base_sql, params={'rcid':RCID, 'country':US_JOB,
    'junior':tuple(JUNIOR), 'start_from':START_FROM.date(),
    'start_before':START_BEFORE.date()}, coerce_float=False)
if base.empty:
    raise ValueError('No candidates. Check access, company, dates and codes.')
base.to_csv(OUT/'google_2018_candidates_raw.csv', index=False)
# Deduplicate only the lookup list, never the raw position records here.
ids = base['user_id'].dropna().astype(str).unique().tolist()
(OUT/'user_ids.txt').write_text('\n'.join(ids))


In [ ]:
def fetch_users(table, mapping, ids, extra='', extra_params=None):
    # Retrieve manageable batches, with IDs passed as query parameters.
    parts = []
    for i in range(0,len(ids),1000):
        sql = (f'SELECT {select_columns(mapping)} FROM {qualified(table)} '
               f'WHERE {ident(mapping["user_id"])} IN %(ids)s {extra}')
        params = {'ids':tuple(ids[i:i+1000]), **(extra_params or {})}
        parts.append(db.raw_sql(sql, params=params, coerce_float=False))
    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(columns=mapping)

# All education for candidate users; education eligibility is evaluated later.
education = fetch_users(EDU_TABLE,E,ids)
education.to_csv(OUT/'education_raw.csv',index=False)
# ALL employers, ALL seniorities, ALL countries. No lower date bound, because
# pre-2018 records are needed to distinguish new Google entrants from transfers.
# Keep unknown starts for review, instead of silently losing incomplete history.
restriction = (f'AND ({ident(P["startdate"])} <= %(through)s '
               f'OR {ident(P["startdate"])} IS NULL)')
positions = fetch_users(POS_TABLE,P,ids,restriction,{'through':EXTRACT_THROUGH.date()})
positions.to_csv(OUT/'positions_all_employers_raw.csv',index=False)
# Save configuration/provenance so the extract can be reproduced.
manifest = {'design':'2018 Google entry/junior candidates; all-employer follow-up',
    'start_from':str(START_FROM.date()), 'start_before':str(START_BEFORE.date()),
    'extract_through':str(EXTRACT_THROUGH.date()), 'followup_years':FOLLOWUP_YEARS,
    'google_rcid':str(RCID), 'junior_codes':JUNIOR, 'baseline_country':US_JOB,
    'history_coverage_confirmed':HISTORY_COVERAGE_CONFIRMED,
    'library':LIB, 'positions_table':POS_TABLE, 'education_table':EDU_TABLE,
    'position_mapping':P, 'education_mapping':E,
    'extracted_at_utc':pd.Timestamp.now(tz='UTC').isoformat()}
(OUT/'extraction_manifest.json').write_text(json.dumps(manifest,indent=2))
db.close()
print(f'Extraction complete. Files saved in {OUT.resolve()}')
# EXTRACTION ENDS HERE. No preprocessing, EDA or summary statistics above.
